In [12]:
from langgraph.graph  import StateGraph,START,END
from langchain_groq import ChatGroq
from langchain_core.messages import BaseMessage,HumanMessage
from typing import TypedDict,Annotated,Literal
from langgraph.checkpoint.memory import MemorySaver
# from pyndatic import BaseModel,Field
import dotenv



In [13]:
from langgraph.graph import add_messages
class ChatState(TypedDict):
    messages:Annotated[list[dict[BaseMessage]],add_messages]

In [ ]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.9
)
def chat_node(state:ChatState)->ChatState:
    messages= state['messages']
    response=llm.invoke(messages).content
    
 
    return {'messages':[response]}

In [17]:
checkpointer=MemorySaver()
graph=StateGraph(ChatState)
graph.add_node('chat_node',chat_node)
graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)
chatbot=graph.compile(checkpointer=checkpointer)
# initial_state={
#   "messages": [HumanMessage(content="whats is the capital of India")]
# }
# result=chatbot.invoke(initial_state)
# result


In [19]:
thread_id = "1"

while True:
  user_message=input("Type here:")
  if user_message.lower() in ['exit','quit','bye']:
    print("Chat Ended")
    break
  config={'configurable':{'thread_id':thread_id}}
  response = chatbot.invoke({'messages':[HumanMessage(content=user_message)]},config=config)
  print("Bot:",response['messages'][-1].content)

Bot: How can I assist you today?
Bot: Nice to meet you, Sneha. What's on your mind today?
Bot: Your name is Sneha!
Bot: It's nice to chat with you on this lovely Friday, Sneha. What are your plans for the weekend so far?
Chat Ended
